In [ ]:
import os

CARPETA_DOCUMENTOS = r"C:\SERCOP_FINAL\05_documentos"

print(f"¿Existe {CARPETA_DOCUMENTOS}?: {os.path.isdir(CARPETA_DOCUMENTOS)}\n")

if os.path.isdir(CARPETA_DOCUMENTOS):
    print("Contenido de 05_documentos:")
    for nombre in os.listdir(CARPETA_DOCUMENTOS):
        ruta_completa = os.path.join(CARPETA_DOCUMENTOS, nombre)
        tipo = "carpeta" if os.path.isdir(ruta_completa) else "archivo"
        print(f"  - '{nombre}'  ({tipo})")

In [ ]:
import pandas as pd
import os

CARPETA_SRI = r"C:\SERCOP_FINAL\05_documentos\SRI"
RUTA_FANTASMAS = r"C:\SERCOP_FINAL\05_documentos\SRI_Catastro_Empresas_Fantasmas.csv"

# --- Listar archivos de RUC ---
archivos_sri = [f for f in os.listdir(CARPETA_SRI) if f.endswith(".xlsx")]
print(f"Archivos de catastro RUC encontrados: {len(archivos_sri)}")
for f in sorted(archivos_sri):
    tam_kb = os.path.getsize(os.path.join(CARPETA_SRI, f)) / 1024
    print(f"  - {f:35s}  {tam_kb:10.1f} KB")

In [ ]:
print(f"¿Existe {CARPETA_SRI}?: {os.path.isdir(CARPETA_SRI)}\n")

if os.path.isdir(CARPETA_SRI):
    contenido = os.listdir(CARPETA_SRI)
    print(f"Elementos encontrados: {len(contenido)}")
    for nombre in contenido:
        ruta_completa = os.path.join(CARPETA_SRI, nombre)
        if os.path.isfile(ruta_completa):
            tam_kb = os.path.getsize(ruta_completa) / 1024
            print(f"  - '{nombre}'  ({tam_kb:.1f} KB)")
        else:
            print(f"  - '{nombre}'  (carpeta)")

In [ ]:
import duckdb
import os

con = duckdb.connect(database=':memory:')

CARPETA_SRI = r"C:\SERCOP_FINAL\05_documentos\SRI"
archivo_prueba_sri = os.path.join(CARPETA_SRI, "SRI_RUC_Galapagos.csv").replace("\\", "/")

# --- Primero, las primeras líneas crudas para ver el delimitador y encoding ---
with open(archivo_prueba_sri.replace("/", "\\"), 'rb') as f:
    primeras_lineas_bytes = [next(f) for _ in range(3)]

for encoding in ['utf-8', 'latin-1', 'cp1252']:
    print(f"--- Probando encoding: {encoding} ---")
    try:
        for linea in primeras_lineas_bytes:
            print(linea.decode(encoding)[:200])
    except Exception as e:
        print(f"Error: {e}")
    print()

In [ ]:
# --- Estructura completa y conteo del archivo de prueba ---
describe_sri = con.execute(f"""
    DESCRIBE
    SELECT * FROM read_csv('{archivo_prueba_sri}', delim='|', header=True, encoding='utf-8')
""").fetchdf()
print(describe_sri.to_string(index=False))

In [ ]:
n_filas_sri = con.execute(f"""
    SELECT COUNT(*) FROM read_csv('{archivo_prueba_sri}', delim='|', header=True, encoding='utf-8')
""").fetchone()[0]
print(f"Filas en Galápagos: {n_filas_sri:,}")

muestra_sri = con.execute(f"""
    SELECT * FROM read_csv('{archivo_prueba_sri}', delim='|', header=True, encoding='utf-8') LIMIT 3
""").fetchdf()
print(muestra_sri.to_string())

In [ ]:
# --- ¿Cuántos NUMERO_RUC distintos hay? (¿una fila por RUC, o por establecimiento?) ---
n_ruc_distintos = con.execute(f"""
    SELECT COUNT(*) AS n_filas, COUNT(DISTINCT NUMERO_RUC) AS n_ruc_distintos
    FROM read_csv('{archivo_prueba_sri}', delim='|', header=True, encoding='utf-8')
""").fetchdf()
print(n_ruc_distintos.to_string(index=False))

In [ ]:
# --- ¿RAZON_SOCIAL y ESTADO_CONTRIBUYENTE son iguales entre establecimientos del mismo RUC? ---
consistencia = con.execute(f"""
    SELECT
        COUNT(DISTINCT NUMERO_RUC) AS n_ruc,
        SUM(CASE WHEN n_razon_social > 1 THEN 1 ELSE 0 END) AS n_ruc_con_razon_social_distinta,
        SUM(CASE WHEN n_estado > 1 THEN 1 ELSE 0 END) AS n_ruc_con_estado_distinto
    FROM (
        SELECT NUMERO_RUC,
               COUNT(DISTINCT RAZON_SOCIAL) AS n_razon_social,
               COUNT(DISTINCT ESTADO_CONTRIBUYENTE) AS n_estado
        FROM read_csv('{archivo_prueba_sri}', delim='|', header=True, encoding='utf-8')
        GROUP BY NUMERO_RUC
    )
""").fetchdf()
print(consistencia.to_string(index=False))

# --- ¿NUMERO_ESTABLECIMIENTO = 1 existe siempre como "matriz"? ---
tiene_matriz = con.execute(f"""
    SELECT
        COUNT(DISTINCT NUMERO_RUC) AS n_ruc_total,
        COUNT(DISTINCT CASE WHEN NUMERO_ESTABLECIMIENTO = 1 THEN NUMERO_RUC END) AS n_ruc_con_matriz
    FROM read_csv('{archivo_prueba_sri}', delim='|', header=True, encoding='utf-8')
""").fetchdf()
print(tiene_matriz.to_string(index=False))

In [ ]:
import time

CARPETA_SRI_PARQUET = r"C:\SERCOP_FINAL\03_resultados"
archivos_sri_todos = [f for f in os.listdir(CARPETA_SRI) if f.startswith("SRI_RUC_") and f.endswith(".csv")]
print(f"Provincias a procesar: {len(archivos_sri_todos)}")

resumen_sri = []

for archivo in sorted(archivos_sri_todos):
    ruta = os.path.join(CARPETA_SRI, archivo).replace("\\", "/")
    provincia = archivo.replace("SRI_RUC_", "").replace(".csv", "")

    t0 = time.time()
    n_total = con.execute(f"""
        SELECT COUNT(*) FROM read_csv('{ruta}', delim='|', header=True, encoding='utf-8')
    """).fetchone()[0]
    n_matriz = con.execute(f"""
        SELECT COUNT(*) FROM read_csv('{ruta}', delim='|', header=True, encoding='utf-8')
        WHERE NUMERO_ESTABLECIMIENTO = 1
    """).fetchone()[0]
    t1 = time.time()

    resumen_sri.append({"provincia": provincia, "n_filas_total": n_total, "n_ruc_matriz": n_matriz})
    print(f"{provincia:20s}  {n_total:>10,} filas  ->  {n_matriz:>10,} RUC  ({t1-t0:.1f}s)")

print(f"\nTotal RUC (matriz) en las 24 provincias: {sum(r['n_ruc_matriz'] for r in resumen_sri):,}")

In [ ]:
CARPETA_RESULTADOS = r"C:\SERCOP_FINAL\03_resultados"

In [ ]:
selects_sri = []
for archivo in sorted(archivos_sri_todos):
    ruta = os.path.join(CARPETA_SRI, archivo).replace("\\", "/")
    provincia = archivo.replace("SRI_RUC_", "").replace(".csv", "")
    selects_sri.append(f"""
        SELECT
            NUMERO_RUC,
            RAZON_SOCIAL,
            ESTADO_CONTRIBUYENTE,
            CLASE_CONTRIBUYENTE,
            TIPO_CONTRIBUYENTE,
            FECHA_INICIO_ACTIVIDADES,
            FECHA_ACTUALIZACION,
            FECHA_SUSPENSION_DEFINITIVA,
            FECHA_REINICIO_ACTIVIDADES,
            DESCRIPCION_PROVINCIA_EST,
            CODIGO_CIIU,
            ACTIVIDAD_ECONOMICA,
            '{provincia}' AS provincia_archivo
        FROM read_csv('{ruta}', delim='|', header=True, encoding='utf-8')
        WHERE NUMERO_ESTABLECIMIENTO = 1
    """)

query_sri_consolidado = " UNION ALL BY NAME ".join(selects_sri)

RUTA_SRI_CATASTRO = os.path.join(CARPETA_RESULTADOS, "sri_ruc_catastro.parquet").replace("\\", "/")

t0 = time.time()
con.execute(f"""
    COPY ({query_sri_consolidado})
    TO '{RUTA_SRI_CATASTRO}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
t1 = time.time()

n_sri_total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_SRI_CATASTRO}')").fetchone()[0]
n_ruc_unicos_sri = con.execute(f"SELECT COUNT(DISTINCT NUMERO_RUC) FROM read_parquet('{RUTA_SRI_CATASTRO}')").fetchone()[0]

print(f"Consolidado en {t1-t0:.1f}s")
print(f"sri_ruc_catastro.parquet: {n_sri_total:,} filas, {n_ruc_unicos_sri:,} RUC únicos")

assert n_sri_total == sum(r['n_ruc_matriz'] for r in resumen_sri)
print("OK: coincide con la suma por provincia.")

In [ ]:
# --- Verificar si hay RUC duplicados entre provincias (un mismo RUC apareciendo en 2 archivos) ---
duplicados_entre_provincias = con.execute(f"""
    SELECT NUMERO_RUC, COUNT(*) AS n
    FROM read_parquet('{RUTA_SRI_CATASTRO}')
    GROUP BY 1
    HAVING COUNT(*) > 1
    ORDER BY n DESC
    LIMIT 10
""").fetchdf()
print(f"RUC duplicados entre provincias: {len(duplicados_entre_provincias)}")
if len(duplicados_entre_provincias) > 0:
    print(duplicados_entre_provincias.to_string(index=False))

In [ ]:
RUTA_V9 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v9.parquet").replace("\\", "/")
RUTA_SRI_CATASTRO = os.path.join(CARPETA_RESULTADOS, "sri_ruc_catastro.parquet").replace("\\", "/")

# --- ¿Qué % de entidad_ruc / proveedor_ruc encuentran match en el catastro del SRI? ---
cobertura = con.execute(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN entidad_ruc IS NOT NULL THEN 1 ELSE 0 END) AS n_con_entidad_ruc,
        SUM(CASE WHEN e.NUMERO_RUC IS NOT NULL THEN 1 ELSE 0 END) AS n_entidad_match_sri,
        SUM(CASE WHEN proveedor_ruc IS NOT NULL THEN 1 ELSE 0 END) AS n_con_proveedor_ruc,
        SUM(CASE WHEN p.NUMERO_RUC IS NOT NULL THEN 1 ELSE 0 END) AS n_proveedor_match_sri
    FROM read_parquet('{RUTA_V9}') b
    LEFT JOIN read_parquet('{RUTA_SRI_CATASTRO}') e ON b.entidad_ruc = e.NUMERO_RUC
    LEFT JOIN read_parquet('{RUTA_SRI_CATASTRO}') p ON b.proveedor_ruc = p.NUMERO_RUC
""").fetchdf()
print(cobertura.to_string(index=False))

In [ ]:
query_antiguedad = f"""
    SELECT
        b.*,
        p.FECHA_INICIO_ACTIVIDADES AS proveedor_fecha_inicio_actividades,
        p.ESTADO_CONTRIBUYENTE AS proveedor_estado_actual_sri,
        p.RAZON_SOCIAL AS proveedor_razon_social_sri,
        DATE_DIFF('day', p.FECHA_INICIO_ACTIVIDADES, b.fecha_release) AS proveedor_antiguedad_dias
    FROM read_parquet('{RUTA_V9}') b
    LEFT JOIN read_parquet('{RUTA_SRI_CATASTRO}') p ON b.proveedor_ruc = p.NUMERO_RUC
"""

n_check_ant = con.execute(f"SELECT COUNT(*) FROM ({query_antiguedad})").fetchone()[0]
print(f"Filas resultantes: {n_check_ant:,}  (debería ser 2,736,387)")

diag_antiguedad = con.execute(f"""
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN proveedor_antiguedad_dias IS NULL THEN 1 ELSE 0 END) AS n_nulo,
        SUM(CASE WHEN proveedor_antiguedad_dias < 0 THEN 1 ELSE 0 END) AS n_negativo,
        MIN(proveedor_antiguedad_dias) AS min_dias,
        MEDIAN(proveedor_antiguedad_dias) AS mediana_dias,
        MAX(proveedor_antiguedad_dias) AS max_dias,
        SUM(CASE WHEN proveedor_antiguedad_dias BETWEEN 0 AND 90 THEN 1 ELSE 0 END) AS n_menos_90dias,
        SUM(CASE WHEN proveedor_antiguedad_dias BETWEEN 0 AND 30 THEN 1 ELSE 0 END) AS n_menos_30dias
    FROM ({query_antiguedad})
""").fetchdf()
print(diag_antiguedad.to_string(index=False))

In [ ]:
# --- ¿Qué está pasando con las antigüedades extremas? ---
extremos_altos = con.execute(f"""
    SELECT proveedor_ruc, proveedor_razon_social_sri, proveedor_fecha_inicio_actividades,
           fecha_release, proveedor_antiguedad_dias
    FROM ({query_antiguedad})
    WHERE proveedor_antiguedad_dias IS NOT NULL
    ORDER BY proveedor_antiguedad_dias DESC
    LIMIT 10
""").fetchdf()
print("Antigüedades más altas (sospechosas):")
print(extremos_altos.to_string(index=False))

print()

# --- Y los 18 casos negativos (RUC creado DESPUÉS del proceso) ---
extremos_negativos = con.execute(f"""
    SELECT proveedor_ruc, proveedor_razon_social_sri, proveedor_fecha_inicio_actividades,
           fecha_release, proveedor_antiguedad_dias, tipo_contratacion_canonico, monto_final
    FROM ({query_antiguedad})
    WHERE proveedor_antiguedad_dias < 0
    ORDER BY proveedor_antiguedad_dias ASC
""").fetchdf()
print("Casos con antigüedad negativa:")
print(extremos_negativos.to_string(index=False))

In [ ]:
tipos_contribuyente = con.execute(f"""
    SELECT p.TIPO_CONTRIBUYENTE, COUNT(*) AS n
    FROM read_parquet('{RUTA_SRI_CATASTRO}') p
    GROUP BY 1
    ORDER BY n DESC
""").fetchdf()
print(tipos_contribuyente.to_string(index=False))

In [ ]:
# --- Identificar RUC de entidades públicas: cualquiera que alguna vez actuó como comprador en SERCOP ---
query_marca_publico = f"""
    WITH rucs_publicos AS (
        SELECT DISTINCT entidad_ruc AS ruc FROM read_parquet('{RUTA_V9}') WHERE entidad_ruc IS NOT NULL
    )
    SELECT
        a.*,
        (rp.ruc IS NOT NULL) AS proveedor_es_entidad_publica
    FROM ({query_antiguedad}) a
    LEFT JOIN rucs_publicos rp ON a.proveedor_ruc = rp.ruc
"""

diag_publico = con.execute(f"""
    SELECT
        proveedor_es_entidad_publica,
        COUNT(*) AS n,
        MEDIAN(proveedor_antiguedad_dias) AS mediana_dias,
        MAX(proveedor_antiguedad_dias) AS max_dias
    FROM ({query_marca_publico})
    GROUP BY 1
""").fetchdf()
print(diag_publico.to_string(index=False))

# --- Verificar que los 3 casos extremos que vimos SÍ quedan marcados como entidad pública ---
check_casos = con.execute(f"""
    SELECT DISTINCT proveedor_ruc, proveedor_razon_social_sri, proveedor_es_entidad_publica
    FROM ({query_marca_publico})
    WHERE proveedor_ruc IN ('RUC_OCULTO_S1', '0360000230001', '0160001240001')
""").fetchdf()
print("\nVerificación de los 3 casos extremos:")
print(check_casos.to_string(index=False))

In [ ]:
query_antiguedad_final = f"""
    SELECT
        * EXCLUDE (proveedor_antiguedad_dias),
        CASE WHEN NOT proveedor_es_entidad_publica THEN proveedor_antiguedad_dias ELSE NULL END AS proveedor_antiguedad_dias
    FROM ({query_marca_publico})
"""

RUTA_V10 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v10.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_antiguedad_final})
    TO '{RUTA_V10}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v10 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V10}')").fetchone()[0]
print(f"base_modelado_v10.parquet: {n_v10:,} filas — OK" if n_v10 == 2736387 else f"OJO: {n_v10:,} filas")

diag_final = con.execute(f"""
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN proveedor_antiguedad_dias IS NOT NULL THEN 1 ELSE 0 END) AS n_elegibles,
        SUM(CASE WHEN proveedor_antiguedad_dias BETWEEN 0 AND 90 THEN 1 ELSE 0 END) AS n_menos_90dias,
        SUM(CASE WHEN proveedor_antiguedad_dias BETWEEN 0 AND 30 THEN 1 ELSE 0 END) AS n_menos_30dias,
        SUM(CASE WHEN proveedor_antiguedad_dias < 0 THEN 1 ELSE 0 END) AS n_negativo
    FROM read_parquet('{RUTA_V10}')
""").fetchdf()
print(diag_final.to_string(index=False))

In [ ]:
RUTA_FANTASMAS = r"C:\SERCOP_FINAL\05_documentos\SRI_Catastro_Empresas_Fantasmas.csv"

print(f"Tamaño: {os.path.getsize(RUTA_FANTASMAS) / 1024:.1f} KB\n")

with open(RUTA_FANTASMAS, encoding='utf-8', errors='replace') as f:
    for i, linea in enumerate(f):
        print(linea.strip()[:300])
        if i >= 5:
            break

In [ ]:
with open(RUTA_FANTASMAS, 'rb') as f:
    lineas_bytes = [next(f) for _ in range(5)]

for encoding in ['latin-1', 'cp1252', 'utf-8']:
    print(f"--- {encoding} ---")
    try:
        for linea in lineas_bytes[3:4]:  # la línea con "COROZO ESTUPIÑAN"
            print(linea.decode(encoding))
    except Exception as e:
        print(f"Error: {e}")
    print()

In [ ]:
with open(RUTA_FANTASMAS, 'rb') as f:
    lineas_bytes = [next(f) for _ in range(6)]

# La línea 4 (índice 4, "COROZO ESTUPIÑAN") es la que tiene el caracter especial
for encoding in ['latin-1', 'cp1252', 'utf-8']:
    print(f"--- {encoding} ---")
    try:
        print(lineas_bytes[4].decode(encoding))
    except Exception as e:
        print(f"Error: {e}")
    print()

In [ ]:
import pandas as pd

df_fantasmas = pd.read_csv(RUTA_FANTASMAS, sep='|', encoding='latin-1')

print(f"Dimensiones: {df_fantasmas.shape[0]:,} filas x {df_fantasmas.shape[1]} columnas")
print(f"\nColumnas: {list(df_fantasmas.columns)}")
print(f"\nTipos de dato:")
print(df_fantasmas.dtypes)

In [ ]:
# --- Distribución de ESTADO_RUC y TIPO_CONTRIBUYENTE ---
print(df_fantasmas['ESTADO_RUC'].value_counts())
print()
print(df_fantasmas['TIPO_CONTRIBUYENTE'].value_counts())
print()
# --- ¿NUMERO_RUC es único, o puede repetirse (varias calificaciones para el mismo RUC)? ---
print(f"Filas totales: {len(df_fantasmas):,}")
print(f"RUC únicos: {df_fantasmas['NUMERO_RUC'].nunique():,}")

In [ ]:
# --- Reconstruir NUMERO_RUC como texto de 13 dígitos, sin perder ceros a la izquierda ---
df_fantasmas['numero_ruc_str'] = df_fantasmas['NUMERO_RUC'].apply(
    lambda x: str(int(x)).zfill(13) if pd.notna(x) else None
)

print("Muestra de conversión RUC:")
print(df_fantasmas[['NUMERO_RUC', 'numero_ruc_str']].head(5).to_string(index=False))

# --- Diagnóstico: ¿qué RUC tienen más de una fila, y por qué? ---
ruc_duplicados = df_fantasmas[df_fantasmas.duplicated('numero_ruc_str', keep=False)].sort_values('numero_ruc_str')
print(f"\nRUC con más de una fila: {ruc_duplicados['numero_ruc_str'].nunique()}")
print(ruc_duplicados[['numero_ruc_str', ' RAZON_SOCIAL', 'ESTADO_RUC', 'DETALLE_CALIFICACION']].head(10).to_string(index=False))

In [ ]:
n_ruc_nulo = df_fantasmas['NUMERO_RUC'].isna().sum()
print(f"Filas con NUMERO_RUC vacío: {n_ruc_nulo}")

# --- Duplicados reales (excluyendo los nulos) ---
solo_con_ruc = df_fantasmas[df_fantasmas['numero_ruc_str'].notna()]
ruc_duplicados_reales = solo_con_ruc[solo_con_ruc.duplicated('numero_ruc_str', keep=False)].sort_values('numero_ruc_str')

print(f"Filas con RUC válido: {len(solo_con_ruc):,}")
print(f"RUC únicos (con RUC válido): {solo_con_ruc['numero_ruc_str'].nunique():,}")
print(f"RUC con más de una fila (reales): {ruc_duplicados_reales['numero_ruc_str'].nunique()}")

if len(ruc_duplicados_reales) > 0:
    print(ruc_duplicados_reales[['numero_ruc_str', ' RAZON_SOCIAL', 'ESTADO_RUC', 'DETALLE_CALIFICACION']].head(10).to_string(index=False))

In [ ]:
# --- Registrar el DataFrame como tabla en DuckDB para poder cruzarlo con SQL ---
fantasmas_validos = df_fantasmas[df_fantasmas['numero_ruc_str'].notna()][
    ['numero_ruc_str', ' RAZON_SOCIAL', 'TIPO_CONTRIBUYENTE', 'ESTADO_RUC', 'DETALLE_CALIFICACION']
].rename(columns={' RAZON_SOCIAL': 'razon_social_fantasma'})

con.register('empresas_fantasmas', fantasmas_validos)

# --- Cruce contra proveedor_ruc y entidad_ruc ---
query_fantasmas = f"""
    SELECT
        b.*,
        (pf.numero_ruc_str IS NOT NULL) AS proveedor_es_fantasma,
        pf.estado_ruc AS proveedor_fantasma_estado,
        pf.detalle_calificacion AS proveedor_fantasma_detalle,
        (ef.numero_ruc_str IS NOT NULL) AS entidad_es_fantasma
    FROM read_parquet('{RUTA_V10}') b
    LEFT JOIN empresas_fantasmas pf ON b.proveedor_ruc = pf.numero_ruc_str
    LEFT JOIN empresas_fantasmas ef ON b.entidad_ruc = ef.numero_ruc_str
"""

resumen_fantasmas = con.execute(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS n_procesos_proveedor_fantasma,
        COUNT(DISTINCT CASE WHEN proveedor_es_fantasma THEN proveedor_ruc END) AS n_proveedores_fantasma_distintos,
        SUM(CASE WHEN entidad_es_fantasma THEN 1 ELSE 0 END) AS n_procesos_entidad_fantasma
    FROM ({query_fantasmas})
""").fetchdf()
print(resumen_fantasmas.to_string(index=False))

In [ ]:
RUTA_V11 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v11.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_fantasmas})
    TO '{RUTA_V11}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v11 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V11}')").fetchone()[0]
print(f"base_modelado_v11.parquet: {n_v11:,} filas — OK" if n_v11 == 2736387 else f"OJO: {n_v11:,}")

In [ ]:
# --- ¿Estos 170 casos ya muestran señales en los indicadores que construimos? ---
casos_fantasma = con.execute(f"""
    SELECT
        proveedor_ruc, proveedor_nombre, entidad_nombre, tipo_contratacion_canonico,
        monto_final, fecha_release,
        ROUND(desviacion_precio_log_z, 2) AS desviacion_precio_log_z,
        proveedor_unico,
        n_contratos_30d_no_catalogo,
        ROUND(concentracion_entidad_proveedor, 3) AS concentracion_entidad_proveedor,
        proveedor_antiguedad_dias
    FROM read_parquet('{RUTA_V11}')
    WHERE proveedor_es_fantasma
    ORDER BY fecha_release
""").fetchdf()
print(casos_fantasma.to_string(index=False))

In [ ]:
tally_fantasmas = con.execute(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN ABS(desviacion_precio_log_z) > 3.5 THEN 1 ELSE 0 END) AS n_capturado_por_precio,
        SUM(CASE WHEN proveedor_unico THEN 1 ELSE 0 END) AS n_proveedor_unico_true,
        SUM(CASE WHEN concentracion_entidad_proveedor >= 0.5 THEN 1 ELSE 0 END) AS n_capturado_por_concentracion,
        SUM(CASE WHEN proveedor_antiguedad_dias < 90 THEN 1 ELSE 0 END) AS n_capturado_por_antiguedad,
        SUM(CASE WHEN n_contratos_30d_no_catalogo >= 3 THEN 1 ELSE 0 END) AS n_capturado_por_fragmentacion,
        COUNT(DISTINCT proveedor_ruc) AS n_proveedores_distintos,
        COUNT(DISTINCT entidad_ruc) AS n_entidades_distintas_afectadas
    FROM read_parquet('{RUTA_V11}')
    WHERE proveedor_es_fantasma
""").fetchdf()
print(tally_fantasmas.to_string(index=False))

In [ ]:
query_diversidad = f"""
    SELECT
        *,
        COUNT(DISTINCT entidad_ruc) OVER (PARTITION BY proveedor_ruc) AS n_entidades_distintas_proveedor
    FROM read_parquet('{RUTA_V11}')
"""

# --- ¿Los proveedores fantasma tienen más diversidad de entidades que el resto? ---
comparacion_diversidad = con.execute(f"""
    SELECT
        proveedor_es_fantasma,
        COUNT(DISTINCT proveedor_ruc) AS n_proveedores,
        MEDIAN(n_entidades_distintas_proveedor) AS mediana_n_entidades,
        AVG(n_entidades_distintas_proveedor) AS promedio_n_entidades,
        MAX(n_entidades_distintas_proveedor) AS max_n_entidades
    FROM ({query_diversidad})
    WHERE proveedor_ruc IS NOT NULL
    GROUP BY 1
""").fetchdf()
print(comparacion_diversidad.to_string(index=False))

In [ ]:
query_diversidad_v2 = f"""
    WITH no_catalogo AS (
        SELECT * FROM read_parquet('{RUTA_V11}')
        WHERE tipo_contratacion_grp NOT LIKE 'Catálogo electrónico%'
          AND proveedor_ruc IS NOT NULL
    ),
    con_diversidad AS (
        SELECT
            proveedor_ruc,
            COUNT(DISTINCT entidad_ruc) AS n_entidades_distintas_no_catalogo
        FROM no_catalogo
        GROUP BY 1
    )
    SELECT
        b.*,
        d.n_entidades_distintas_no_catalogo
    FROM read_parquet('{RUTA_V11}') b
    LEFT JOIN con_diversidad d ON b.proveedor_ruc = d.proveedor_ruc
"""

comparacion_v2 = con.execute(f"""
    SELECT
        proveedor_es_fantasma,
        COUNT(DISTINCT proveedor_ruc) AS n_proveedores,
        MEDIAN(n_entidades_distintas_no_catalogo) AS mediana_n_entidades,
        AVG(n_entidades_distintas_no_catalogo) AS promedio_n_entidades,
        MAX(n_entidades_distintas_no_catalogo) AS max_n_entidades
    FROM ({query_diversidad_v2})
    WHERE proveedor_ruc IS NOT NULL
    GROUP BY 1
""").fetchdf()
print(comparacion_v2.to_string(index=False))

In [ ]:
distribucion_tipo_fantasma = con.execute(f"""
    SELECT
        (tipo_contratacion_grp LIKE 'Catálogo electrónico%') AS es_catalogo,
        COUNT(*) AS n_procesos,
        COUNT(DISTINCT proveedor_ruc) AS n_proveedores
    FROM read_parquet('{RUTA_V11}')
    WHERE proveedor_es_fantasma
    GROUP BY 1
""").fetchdf()
print(distribucion_tipo_fantasma.to_string(index=False))

In [ ]:
# ¿Qué tablas existen actualmente en memoria?
print(con.execute("SHOW TABLES").fetchdf())

# ¿Existe el archivo parquet en disco?
import os
ruta_v11 = r"C:\SERCOP_FINAL\03_resultados\base_modelado_v11.parquet"
print("Existe v11 en disco:", os.path.exists(ruta_v11))
print("Tamaño (MB):", round(os.path.getsize(ruta_v11)/1024/1024, 1) if os.path.exists(ruta_v11) else "N/A")

In [ ]:
# --- Paso 0: cargar la base más reciente directamente desde disco ---
con.execute(f"""
    CREATE OR REPLACE TABLE base_modelado_v11 AS
    SELECT * FROM read_parquet('{ruta_v11}')
""")

n_check = con.execute("SELECT COUNT(*) FROM base_modelado_v11").fetchone()[0]
print(f"base_modelado_v11 recargada: {n_check:,} filas")

In [ ]:
carpeta = r"C:\SERCOP_FINAL\03_resultados"
for f in sorted(os.listdir(carpeta)):
    if f.startswith("base_modelado"):
        ruta = os.path.join(carpeta, f)
        print(f, "-", round(os.path.getsize(ruta)/1024/1024, 1), "MB")

In [ ]:
cols = con.execute("DESCRIBE base_modelado_v11").fetchdf()
print(cols[cols['column_name'].str.contains('proveedor|fecha', case=False)])

In [ ]:
cols_sri = con.execute("""
    SELECT * FROM read_parquet('C:/SERCOP_FINAL/03_resultados/sri_ruc_catastro.parquet') LIMIT 0
""").fetchdf()
print(list(cols_sri.columns))

In [ ]:
# =========================================================
# INDICADOR 5 (corregido): traer fechas de suspensión/reactivación
# =========================================================

# --- Paso 0: cargar catastro SRI ---
con.execute("""
    CREATE OR REPLACE TABLE sri_catastro AS
    SELECT
        NUMERO_RUC AS ruc,
        FECHA_SUSPENSION_DEFINITIVA AS fecha_suspension_definitiva,
        FECHA_REINICIO_ACTIVIDADES AS fecha_reinicio_actividades
    FROM read_parquet('C:/SERCOP_FINAL/03_resultados/sri_ruc_catastro.parquet')
""")

n_catastro = con.execute("SELECT COUNT(*) FROM sri_catastro").fetchone()[0]
n_ruc_unicos = con.execute("SELECT COUNT(DISTINCT ruc) FROM sri_catastro").fetchone()[0]
print(f"sri_catastro: {n_catastro:,} filas, {n_ruc_unicos:,} RUC únicos")
assert n_catastro == n_ruc_unicos, "ERROR: hay RUC duplicados en el catastro, revisar antes de seguir"

# --- Paso 1: JOIN para traer las fechas al base_modelado_v11 ---
con.execute("""
    CREATE OR REPLACE TABLE base_modelado_v12 AS
    SELECT
        b.*,
        s.fecha_suspension_definitiva AS proveedor_fecha_suspension_definitiva,
        s.fecha_reinicio_actividades AS proveedor_fecha_reinicio_actividades
    FROM base_modelado_v11 b
    LEFT JOIN sri_catastro s
        ON b.proveedor_ruc = s.ruc
""")

n_v12 = con.execute("SELECT COUNT(*) FROM base_modelado_v12").fetchone()[0]
n_v11 = con.execute("SELECT COUNT(*) FROM base_modelado_v11").fetchone()[0]
print(f"base_modelado_v12: {n_v12:,} filas")
assert n_v12 == n_v11, "ERROR: no debe cambiar el número de filas"
print("OK: reconcilia con base_modelado_v11.")

# --- Paso 2: diagnóstico — ¿cuántos procesos ocurrieron con RUC ya suspendido? ---
diag_post_suspension = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE proveedor_fecha_suspension_definitiva IS NOT NULL
              AND fecha_release >= proveedor_fecha_suspension_definitiva
        ) AS n_proceso_con_ruc_ya_suspendido,
        COUNT(*) FILTER (
            WHERE proveedor_fecha_suspension_definitiva IS NOT NULL
        ) AS n_proveedor_tiene_fecha_suspension,
        COUNT(*) AS n_total
    FROM base_modelado_v12
""").fetchdf()
print(diag_post_suspension)

# --- Paso 3: construir los dos indicadores ---
con.execute("""
    CREATE OR REPLACE TABLE base_modelado_v12 AS
    SELECT
        *,
        (proveedor_estado_actual_sri IN ('SUSPENDIDO', 'PASIVO')) AS proveedor_ruc_estado_riesgoso,
        (
            proveedor_fecha_suspension_definitiva IS NOT NULL
            AND fecha_release >= proveedor_fecha_suspension_definitiva
            AND (
                proveedor_fecha_reinicio_actividades IS NULL
                OR fecha_release < proveedor_fecha_reinicio_actividades
            )
        ) AS proveedor_contrato_tras_suspension
    FROM base_modelado_v12
""")

resumen_ind5 = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE proveedor_ruc_estado_riesgoso) AS n_estado_riesgoso,
        COUNT(*) FILTER (WHERE proveedor_contrato_tras_suspension) AS n_contrato_tras_suspension,
        COUNT(*) AS n_total
    FROM base_modelado_v12
""").fetchdf()
print(resumen_ind5)

# --- Paso 4: el cruce clave — ¿este indicador SÍ captura fantasmas? ---
cruce_fantasma_estado = con.execute("""
    SELECT
        proveedor_es_fantasma,
        COUNT(*) FILTER (WHERE proveedor_ruc_estado_riesgoso) AS n_estado_riesgoso,
        COUNT(*) FILTER (WHERE proveedor_contrato_tras_suspension) AS n_tras_suspension,
        COUNT(*) AS n_total
    FROM base_modelado_v12
    WHERE proveedor_es_fantasma IS NOT NULL
    GROUP BY 1
""").fetchdf()
print(cruce_fantasma_estado)

con.execute("""
    COPY base_modelado_v12 TO 'C:/SERCOP_FINAL/03_resultados/base_modelado_v12.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print("Guardado: base_modelado_v12.parquet")

In [ ]:
# =========================================================
# INDICADOR 6: Discordancia geográfica proveedor-entidad
# =========================================================
# Objetivo: variable derivada del cruce RUC-SRI (no un ground-truth
# como proveedor_es_fantasma), candidata real a entrar al modelo.

# --- Paso 0: traer provincia del proveedor desde el catastro ---
con.execute("""
    CREATE OR REPLACE TABLE sri_catastro_geo AS
    SELECT
        NUMERO_RUC AS ruc,
        DESCRIPCION_PROVINCIA_EST AS provincia_proveedor
    FROM read_parquet('C:/SERCOP_FINAL/03_resultados/sri_ruc_catastro.parquet')
""")

# --- Paso 1: revisar qué columna de provincia tenemos del lado de la entidad ---
cols_entidad = con.execute("DESCRIBE base_modelado_v12").fetchdf()
print(cols_entidad[cols_entidad['column_name'].str.contains('entidad|provincia', case=False)])

In [ ]:
# =========================================================
# INDICADOR 6: Discordancia geográfica proveedor-entidad
# =========================================================

con.execute("""
    CREATE OR REPLACE TABLE sri_catastro_geo AS
    SELECT
        NUMERO_RUC AS ruc,
        DESCRIPCION_PROVINCIA_EST AS provincia_registrada
    FROM read_parquet('C:/SERCOP_FINAL/03_resultados/sri_ruc_catastro.parquet')
""")

con.execute("""
    CREATE OR REPLACE TABLE base_modelado_v13 AS
    SELECT
        b.*,
        sp.provincia_registrada AS provincia_proveedor,
        se.provincia_registrada AS provincia_entidad,
        (
            sp.provincia_registrada IS NOT NULL
            AND se.provincia_registrada IS NOT NULL
            AND sp.provincia_registrada <> se.provincia_registrada
        ) AS proveedor_provincia_distinta_entidad
    FROM base_modelado_v12 b
    LEFT JOIN sri_catastro_geo sp ON b.proveedor_ruc = sp.ruc
    LEFT JOIN sri_catastro_geo se ON b.entidad_ruc = se.ruc
""")

n_v13 = con.execute("SELECT COUNT(*) FROM base_modelado_v13").fetchone()[0]
n_v12 = con.execute("SELECT COUNT(*) FROM base_modelado_v12").fetchone()[0]
print(f"base_modelado_v13: {n_v13:,} filas")
assert n_v13 == n_v12, "ERROR: no debe cambiar el número de filas"
print("OK: reconcilia con base_modelado_v12.")

diag_geo = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE provincia_proveedor IS NOT NULL AND provincia_entidad IS NOT NULL) AS n_con_ambas_provincias,
        COUNT(*) FILTER (WHERE proveedor_provincia_distinta_entidad) AS n_provincia_distinta,
        COUNT(*) AS n_total
    FROM base_modelado_v13
""").fetchdf()
print(diag_geo)

cruce_fantasma_geo = con.execute("""
    SELECT
        proveedor_es_fantasma,
        COUNT(*) FILTER (WHERE proveedor_provincia_distinta_entidad) AS n_provincia_distinta,
        COUNT(*) AS n_total
    FROM base_modelado_v13
    WHERE proveedor_es_fantasma IS NOT NULL
    GROUP BY 1
""").fetchdf()
print(cruce_fantasma_geo)

con.execute("""
    COPY base_modelado_v13 TO 'C:/SERCOP_FINAL/03_resultados/base_modelado_v13.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print("Guardado: base_modelado_v13.parquet")

In [ ]:
# ¿Tenemos alguna columna relacionada a estado del contrato/incumplimiento
# que ya hayamos extraído del JSON original?
cols_estado = con.execute("DESCRIBE base_modelado_v13").fetchdf()
print(cols_estado[cols_estado['column_name'].str.contains('estado|status|contract', case=False)])

In [ ]:
# =========================================================
# PASO 0: inventario completo de columnas candidatas para el modelo
# =========================================================

cols_todas = con.execute("DESCRIBE base_modelado_v13").fetchdf()
print(cols_todas.to_string())

In [ ]:
diag_nulos_features = con.execute("""
    SELECT
        COUNT(*) AS n_total,
        COUNT(*) FILTER (WHERE desviacion_precio_log_z IS NULL) AS null_precio,
        COUNT(*) FILTER (WHERE concentracion_entidad_proveedor IS NULL) AS null_concentracion,
        COUNT(*) FILTER (WHERE proveedor_unico IS NULL) AS null_proveedor_unico,
        COUNT(*) FILTER (WHERE proveedor_antiguedad_dias IS NULL) AS null_antiguedad,
        COUNT(*) FILTER (WHERE proveedor_ruc_estado_riesgoso IS NULL) AS null_estado_riesgoso,
        COUNT(*) FILTER (WHERE proveedor_contrato_tras_suspension IS NULL) AS null_tras_suspension,
        COUNT(*) FILTER (WHERE proveedor_provincia_distinta_entidad IS NULL) AS null_provincia_distinta,
        COUNT(*) FILTER (WHERE tiene_periodo_competitivo IS NULL) AS null_periodo_competitivo,
        COUNT(*) FILTER (WHERE proveedor_es_fantasma IS NULL) AS null_es_fantasma
    FROM base_modelado_v13
""").fetchdf()
print(diag_nulos_features.T)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# =========================================================
# MODELO A: Isolation Forest sobre TODOS los procesos
# =========================================================

df_a = con.execute("""
    SELECT
        ocid, proveedor_ruc, entidad_ruc,
        tiene_periodo_competitivo,
        desviacion_precio_log_z,
        concentracion_entidad_proveedor,
        proveedor_antiguedad_dias,
        proveedor_es_entidad_publica,
        proveedor_contrato_tras_suspension,
        proveedor_provincia_distinta_entidad,
        proveedor_es_fantasma
    FROM base_modelado_v13
    WHERE desviacion_precio_log_z IS NOT NULL
      AND concentracion_entidad_proveedor IS NOT NULL
""").fetchdf()

print(f"Filas disponibles para Modelo A: {len(df_a):,}")
print(f"Filas excluidas (nulos en precio/concentración): {2736387 - len(df_a):,}")

# Manejo explícito de nulos en antigüedad (marca, no imputación "típica")
df_a['antiguedad_desconocida'] = df_a['proveedor_antiguedad_dias'].isna().astype(int)
df_a['proveedor_antiguedad_dias_num'] = df_a['proveedor_antiguedad_dias'].fillna(-1)

for col in ['proveedor_contrato_tras_suspension', 'proveedor_provincia_distinta_entidad',
            'tiene_periodo_competitivo', 'proveedor_es_entidad_publica']:
    df_a[col] = df_a[col].astype(int)

features_a = [
    'desviacion_precio_log_z',
    'concentracion_entidad_proveedor',
    'proveedor_antiguedad_dias_num',
    'antiguedad_desconocida',
    'proveedor_es_entidad_publica',
    'proveedor_contrato_tras_suspension',
    'proveedor_provincia_distinta_entidad',
    'tiene_periodo_competitivo',
]

print("\nDescribe de features Modelo A:")
print(df_a[features_a].describe().T)

# --- Entrenamiento ---
X_a = df_a[features_a].values

clf_a = IsolationForest(
    n_estimators=200,
    max_samples='auto',
    contamination='auto',
    random_state=42,
    n_jobs=-1
)
clf_a.fit(X_a)

df_a['anomaly_score_a'] = -clf_a.score_samples(X_a)  # más alto = más anómalo
df_a['es_anomalia_a'] = (clf_a.predict(X_a) == -1)

print(f"\nProcesos marcados como anomalía (contamination='auto'): {df_a['es_anomalia_a'].sum():,} ({df_a['es_anomalia_a'].mean()*100:.2f}%)")

# --- Validación contra empresas fantasma conocidas ---
df_a['percentil_score_a'] = df_a['anomaly_score_a'].rank(pct=True) * 100

resumen_validacion_a = df_a.groupby('proveedor_es_fantasma').agg(
    n=('anomaly_score_a', 'size'),
    media_percentil=('percentil_score_a', 'mean'),
    mediana_percentil=('percentil_score_a', 'median'),
    n_top1pct=('percentil_score_a', lambda s: (s >= 99).sum()),
    n_top5pct=('percentil_score_a', lambda s: (s >= 95).sum()),
    n_top10pct=('percentil_score_a', lambda s: (s >= 90).sum()),
).reset_index()

print("\nValidación Modelo A contra proveedor_es_fantasma:")
print(resumen_validacion_a)

df_a.to_parquet('C:/SERCOP_FINAL/03_resultados/scores_modelo_a.parquet', index=False)
print("\nGuardado: scores_modelo_a.parquet")

In [ ]:
from sklearn.metrics import roc_auc_score

y_true = df_a['proveedor_es_fantasma'].astype(int)
y_score = df_a['anomaly_score_a']

auc_a = roc_auc_score(y_true, y_score)
print(f"AUC-ROC Modelo A: {auc_a:.4f}")

# --- Curva de lift: si revisamos el top X% más anómalo, ¿cuántas fantasmas atrapamos? ---
umbrales = [0.1, 0.5, 1, 2, 5, 10, 20, 30]
resultados_lift = []
for u in umbrales:
    corte = 100 - u
    capturadas = ((df_a['percentil_score_a'] >= corte) & (df_a['proveedor_es_fantasma'])).sum()
    esperado_azar = round(170 * u / 100, 1)
    resultados_lift.append({
        'top_%_revisado': u,
        'fantasmas_capturadas': capturadas,
        'esperado_por_azar': esperado_azar,
        'lift': round(capturadas / esperado_azar, 2) if esperado_azar > 0 else None
    })

df_lift = pd.DataFrame(resultados_lift)
print(df_lift)

In [ ]:
# =========================================================
# MODELO B: Isolation Forest sobre procesos CON periodo competitivo
# =========================================================

df_b = con.execute("""
    SELECT
        ocid, proveedor_ruc, entidad_ruc,
        desviacion_precio_log_z,
        concentracion_entidad_proveedor,
        proveedor_antiguedad_dias,
        proveedor_es_entidad_publica,
        proveedor_contrato_tras_suspension,
        proveedor_provincia_distinta_entidad,
        proveedor_unico,
        n_oferentes,
        proveedor_es_fantasma
    FROM base_modelado_v13
    WHERE tiene_periodo_competitivo = TRUE
      AND desviacion_precio_log_z IS NOT NULL
      AND concentracion_entidad_proveedor IS NOT NULL
""").fetchdf()

print(f"Filas disponibles para Modelo B: {len(df_b):,}")

df_b['antiguedad_desconocida'] = df_b['proveedor_antiguedad_dias'].isna().astype(int)
df_b['proveedor_antiguedad_dias_num'] = df_b['proveedor_antiguedad_dias'].fillna(-1)
df_b['n_oferentes'] = df_b['n_oferentes'].fillna(0)

for col in ['proveedor_contrato_tras_suspension', 'proveedor_provincia_distinta_entidad',
            'proveedor_es_entidad_publica', 'proveedor_unico']:
    df_b[col] = df_b[col].astype(int)

features_b = [
    'desviacion_precio_log_z',
    'concentracion_entidad_proveedor',
    'proveedor_antiguedad_dias_num',
    'antiguedad_desconocida',
    'proveedor_es_entidad_publica',
    'proveedor_contrato_tras_suspension',
    'proveedor_provincia_distinta_entidad',
    'proveedor_unico',
    'n_oferentes',
]

print("\nDescribe de features Modelo B:")
print(df_b[features_b].describe().T)

X_b = df_b[features_b].values
clf_b = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
clf_b.fit(X_b)

df_b['anomaly_score_b'] = -clf_b.score_samples(X_b)
df_b['percentil_score_b'] = df_b['anomaly_score_b'].rank(pct=True) * 100

y_true_b = df_b['proveedor_es_fantasma'].astype(int)
auc_b = roc_auc_score(y_true_b, df_b['anomaly_score_b'])
print(f"\nAUC-ROC Modelo B: {auc_b:.4f}")

resumen_validacion_b = df_b.groupby('proveedor_es_fantasma').agg(
    n=('anomaly_score_b', 'size'),
    media_percentil=('percentil_score_b', 'mean'),
    mediana_percentil=('percentil_score_b', 'median'),
).reset_index()
print(resumen_validacion_b)

df_b.to_parquet('C:/SERCOP_FINAL/03_resultados/scores_modelo_b.parquet', index=False)
print("Guardado: scores_modelo_b.parquet")

In [ ]:
# --- Diagnóstico: ¿cuántos nulos quedan en proveedor_unico dentro del subconjunto competitivo? ---
print("Nulos en proveedor_unico (subconjunto competitivo):", df_b['proveedor_unico'].isna().sum())
print("Nulos en n_oferentes:", df_b['n_oferentes'].isna().sum())
print(df_b[df_b['proveedor_unico'].isna()][['n_oferentes']].describe())

In [ ]:
df_b['proveedor_unico_desconocido'] = df_b['proveedor_unico'].isna().astype(int)
df_b['proveedor_unico'] = df_b['proveedor_unico'].fillna(False).astype(int)

df_b['proveedor_contrato_tras_suspension'] = df_b['proveedor_contrato_tras_suspension'].astype(int)
df_b['proveedor_provincia_distinta_entidad'] = df_b['proveedor_provincia_distinta_entidad'].astype(int)
df_b['proveedor_es_entidad_publica'] = df_b['proveedor_es_entidad_publica'].astype(int)

features_b = [
    'desviacion_precio_log_z',
    'concentracion_entidad_proveedor',
    'proveedor_antiguedad_dias_num',
    'antiguedad_desconocida',
    'proveedor_es_entidad_publica',
    'proveedor_contrato_tras_suspension',
    'proveedor_provincia_distinta_entidad',
    'proveedor_unico',
    'proveedor_unico_desconocido',
    'n_oferentes',
]

print("\nDescribe de features Modelo B:")
print(df_b[features_b].describe().T)

X_b = df_b[features_b].values
clf_b = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
clf_b.fit(X_b)

df_b['anomaly_score_b'] = -clf_b.score_samples(X_b)
df_b['percentil_score_b'] = df_b['anomaly_score_b'].rank(pct=True) * 100

y_true_b = df_b['proveedor_es_fantasma'].astype(int)
auc_b = roc_auc_score(y_true_b, df_b['anomaly_score_b'])
print(f"\nAUC-ROC Modelo B: {auc_b:.4f}")

umbrales = [0.1, 0.5, 1, 2, 5, 10, 20, 30]
resultados_lift_b = []
for u in umbrales:
    corte = 100 - u
    capturadas = ((df_b['percentil_score_b'] >= corte) & (df_b['proveedor_es_fantasma'])).sum()
    n_fantasma_b = df_b['proveedor_es_fantasma'].sum()
    esperado_azar = round(n_fantasma_b * u / 100, 1)
    resultados_lift_b.append({
        'top_%_revisado': u,
        'fantasmas_capturadas': capturadas,
        'esperado_por_azar': esperado_azar,
        'lift': round(capturadas / esperado_azar, 2) if esperado_azar > 0 else None
    })

print(pd.DataFrame(resultados_lift_b))
print(f"\nTotal fantasmas en subconjunto competitivo: {int(y_true_b.sum())}")

df_b.to_parquet('C:/SERCOP_FINAL/03_resultados/scores_modelo_b.parquet', index=False)
print("Guardado: scores_modelo_b.parquet")

In [ ]:
comparacion_b = df_b.groupby('proveedor_es_fantasma')[features_b].agg(['mean', 'median']).T
print(comparacion_b)

# Diferencia relativa por feature (para ver cuál pesa más)
medias = df_b.groupby('proveedor_es_fantasma')[features_b].mean().T
medias.columns = ['no_fantasma', 'fantasma']
medias['diferencia'] = medias['fantasma'] - medias['no_fantasma']
print("\nDiferencias de medias (fantasma - no_fantasma):")
print(medias.sort_values('diferencia', key=abs, ascending=False))

In [ ]:
import duckdb
import json

# Inspeccionar un release completo para ver si trae la lista completa de oferentes
con_tmp = duckdb.connect()
muestra = con_tmp.execute("""
    SELECT r
    FROM (
        SELECT UNNEST(t.releases) AS r
        FROM read_json_auto('C:/SERCOP_FINAL/00_json_crudo/releases_2022.json', format='array') AS t
    )
    WHERE r.tender.numberOfTenderers > 1
    LIMIT 1
""").fetchdf()

registro = muestra['r'].iloc[0]
print("Claves de nivel superior:", list(registro.keys()) if isinstance(registro, dict) else "no es dict")
print("\n¿Tiene 'parties'?:", 'parties' in registro if isinstance(registro, dict) else None)
print("\n¿Tiene 'bids'?:", 'bids' in registro if isinstance(registro, dict) else None)

if isinstance(registro, dict) and 'parties' in registro:
    print("\nEjemplo de parties (primeros 3):")
    for p in registro['parties'][:3]:
        print(p)

if isinstance(registro, dict) and 'bids' in registro:
    print("\nEjemplo de bids:")
    print(registro['bids'])

In [ ]:
import duckdb
import pandas as pd
import json
import os
import time

con2 = duckdb.connect()

carpeta_json = r"C:\SERCOP_FINAL\00_json_crudo"
carpeta_resultados = r"C:\SERCOP_FINAL\03_resultados"

print("Archivos 'oferentes_*' existentes en 03_resultados:")
for f in sorted(os.listdir(carpeta_resultados)):
    if f.startswith("oferentes_"):
        ruta = os.path.join(carpeta_resultados, f)
        tam_mb = round(os.path.getsize(ruta) / 1024 / 1024, 2)
        print(f"  {f} - {tam_mb} MB")

In [ ]:
# Borra los que sean muy pequeños/corruptos (ajusta el umbral si hace falta según lo que salga arriba)
for f in sorted(os.listdir(carpeta_resultados)):
    if f.startswith("oferentes_"):
        ruta = os.path.join(carpeta_resultados, f)
        if os.path.getsize(ruta) < 1000:
            os.remove(ruta)
            print(f"Eliminado: {f}")

In [ ]:
archivos = {
    2015: "releases_2015.json", 2016: "releases_2016.json", 2017: "releases_2017.json",
    2018: "releases_2018.json", 2019: "releases_2019.json", 2020: "releases_2020.json",
    2021: "releases_2021.json", 2022: "releases_2022.json", 2023: "releases_2023.json",
    2024: "releases_2024_reparado.json", 2025: "releases_2025.json", 2026: "releases_2026.json",
}

resumen_anios = []

for anio, archivo in archivos.items():
    ruta_salida = os.path.join(carpeta_resultados, f"oferentes_{anio}.parquet")

    if os.path.exists(ruta_salida) and os.path.getsize(ruta_salida) > 1000:
        n_existente = pd.read_parquet(ruta_salida).shape[0]
        print(f"{anio}: YA EXISTE, {n_existente:,} filas — saltado")
        resumen_anios.append({'anio': anio, 'n_filas': n_existente, 'segundos': 0})
        continue

    ruta = os.path.join(carpeta_json, archivo)
    print(f"{anio}: cargando {archivo} en memoria...", flush=True)
    inicio = time.time()

    with open(ruta, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print(f"   JSON cargado en {time.time()-inicio:.1f}s, {len(data):,} paquetes. Extrayendo oferentes...", flush=True)

    filas = []
    n_releases_procesados = 0

    for paquete in data:
        for release in paquete.get('releases', []):
            ocid = release.get('ocid')
            parties = release.get('parties') or []
            for p in parties:
                roles = p.get('roles') or []
                if 'tenderer' in roles:
                    identifier = p.get('identifier') or {}
                    filas.append({
                        'ocid': ocid,
                        'oferente_id_raw': identifier.get('id'),
                        'oferente_nombre': p.get('name'),
                        'es_ganador': 'supplier' in roles,
                    })
            n_releases_procesados += 1
            if n_releases_procesados % 50000 == 0:
                print(f"   ...{n_releases_procesados:,} releases procesados, {len(filas):,} filas de oferentes hasta ahora", flush=True)

    df_anio = pd.DataFrame(filas)
    df_anio['oferente_ruc'] = df_anio['oferente_id_raw'].str.extract(r'EC-RUC-(\d{13})')
    df_anio.to_parquet(ruta_salida, index=False, compression='zstd')

    dur = time.time() - inicio
    print(f"   -> {anio}: {len(df_anio):,} filas en {dur:.1f}s (total)")
    resumen_anios.append({'anio': anio, 'n_filas': len(df_anio), 'segundos': round(dur, 1)})

print("\nResumen:")
print(pd.DataFrame(resumen_anios))

In [ ]:
# =========================================================
# Consolidar oferentes de los 12 años + diagnóstico de calidad
# =========================================================

dfs = []
suma_por_anio = 0
for anio in archivos.keys():
    ruta = os.path.join(carpeta_resultados, f"oferentes_{anio}.parquet")
    df_anio = pd.read_parquet(ruta)
    df_anio['anio'] = anio
    dfs.append(df_anio)
    suma_por_anio += len(df_anio)

oferentes = pd.concat(dfs, ignore_index=True)
print(f"Consolidado: {len(oferentes):,} filas")
assert len(oferentes) == suma_por_anio, "ERROR: no reconcilia con la suma por año"
print("OK: reconcilia con la suma por año.")

# --- Diagnóstico de calidad ---
print(f"\nOCID únicos (procesos con al menos 1 oferente): {oferentes['ocid'].nunique():,}")
print(f"RUC de oferente únicos: {oferentes['oferente_ruc'].nunique():,}")
print(f"Filas sin RUC extraído: {oferentes['oferente_ruc'].isna().sum():,} ({oferentes['oferente_ruc'].isna().mean()*100:.2f}%)")

# ¿Cuántos oferentes por proceso en promedio? (para comparar contra n_oferentes que ya teníamos)
oferentes_por_proceso = oferentes.groupby('ocid').size()
print(f"\nOferentes por proceso: media={oferentes_por_proceso.mean():.2f}, mediana={oferentes_por_proceso.median():.0f}, max={oferentes_por_proceso.max():,}")
print(oferentes_por_proceso.describe())

oferentes.to_parquet(os.path.join(carpeta_resultados, 'oferentes_consolidado.parquet'), index=False, compression='zstd')
print("\nGuardado: oferentes_consolidado.parquet")

In [ ]:
# --- Percentiles de la cola ---
percentiles = [50, 75, 90, 95, 99, 99.5, 99.9, 100]
print(oferentes_por_proceso.quantile([p/100 for p in percentiles]))

# --- ¿Cuántos procesos superan distintos umbrales, y cuántos pares generarían? ---
from math import comb

umbrales_prueba = [10, 15, 20, 30, 50, 100]
resumen_umbral = []
for u in umbrales_prueba:
    grupos_excluidos = oferentes_por_proceso[oferentes_por_proceso > u]
    n_procesos_excluidos = len(grupos_excluidos)
    n_procesos_incluidos = len(oferentes_por_proceso) - n_procesos_excluidos
    pct_procesos_excluidos = n_procesos_excluidos / len(oferentes_por_proceso) * 100

    # pares que generarían los procesos INCLUIDOS (<=u oferentes)
    grupos_incluidos = oferentes_por_proceso[oferentes_por_proceso <= u]
    pares_incluidos = grupos_incluidos.apply(lambda n: comb(int(n), 2)).sum()

    resumen_umbral.append({
        'umbral_max_oferentes': u,
        'procesos_excluidos': n_procesos_excluidos,
        'pct_procesos_excluidos': round(pct_procesos_excluidos, 3),
        'pares_estimados_incluidos': pares_incluidos,
    })

print(pd.DataFrame(resumen_umbral))

# --- ¿Las 170 fantasmas quedarían dentro o fuera de cada umbral? ---
# (cruce rápido: cuántos oferentes tenían los procesos donde participó un proveedor fantasma)

In [ ]:
con2.execute("""
    CREATE OR REPLACE TABLE base_modelado_v13 AS
    SELECT * FROM read_parquet('C:/SERCOP_FINAL/03_resultados/base_modelado_v13.parquet')
""")

n_check = con2.execute("SELECT COUNT(*) FROM base_modelado_v13").fetchone()[0]
print(f"base_modelado_v13 recargada: {n_check:,} filas")

ocids_fantasma = con2.execute("""
    SELECT DISTINCT ocid FROM base_modelado_v13 WHERE proveedor_es_fantasma = TRUE
""").fetchdf()['ocid']

n_oferentes_fantasma = oferentes_por_proceso.reindex(ocids_fantasma)
print(n_oferentes_fantasma.describe())
print(f"\nFantasmas con más de 30 oferentes en su proceso: {(n_oferentes_fantasma > 30).sum()}")

In [ ]:
from itertools import combinations
from collections import Counter

UMBRAL_MAX_OFERENTES = 30

oferentes_validos = oferentes[oferentes['oferente_ruc'].notna()].copy()
grupos = oferentes_validos.groupby('ocid')['oferente_ruc'].apply(list)
grupos = grupos[grupos.apply(len) <= UMBRAL_MAX_OFERENTES]
grupos = grupos[grupos.apply(len) >= 2]

print(f"Procesos usados para la red de pares: {len(grupos):,}")

contador_pares = Counter()
for lista_rucs in grupos:
    rucs_unicos = sorted(set(lista_rucs))
    for par in combinations(rucs_unicos, 2):
        contador_pares[par] += 1

print(f"Pares únicos encontrados: {len(contador_pares):,}")

df_pares = pd.DataFrame(
    [(a, b, n) for (a, b), n in contador_pares.items()],
    columns=['ruc_a', 'ruc_b', 'n_coincidencias']
).sort_values('n_coincidencias', ascending=False)

print("\nTop 15 pares que más veces compiten juntos:")
print(df_pares.head(15))

print("\nDistribución de n_coincidencias:")
print(df_pares['n_coincidencias'].describe())

df_pares.to_parquet(os.path.join(carpeta_resultados, 'red_coparticipacion.parquet'), index=False, compression='zstd')
print("\nGuardado: red_coparticipacion.parquet")

In [ ]:
# --- Apariciones totales por RUC (dentro del mismo universo de procesos filtrado) ---
apariciones_por_ruc = oferentes_validos[oferentes_validos['ocid'].isin(grupos.index)].groupby('oferente_ruc')['ocid'].nunique()

df_pares = df_pares.merge(apariciones_por_ruc.rename('apariciones_a'), left_on='ruc_a', right_index=True)
df_pares = df_pares.merge(apariciones_por_ruc.rename('apariciones_b'), left_on='ruc_b', right_index=True)

df_pares['jaccard'] = df_pares['n_coincidencias'] / (
    df_pares['apariciones_a'] + df_pares['apariciones_b'] - df_pares['n_coincidencias']
)

print("Top 15 pares por Jaccard (coincidencia relativa, con al menos 3 coincidencias):")
print(df_pares[df_pares['n_coincidencias'] >= 3].sort_values('jaccard', ascending=False).head(15))

# --- RUC de las 15 fantasmas conocidas ---
rucs_fantasma = con2.execute("""
    SELECT DISTINCT proveedor_ruc FROM base_modelado_v13 WHERE proveedor_es_fantasma = TRUE
""").fetchdf()['proveedor_ruc'].tolist()
print(f"\nRUC fantasma: {rucs_fantasma}")

# --- Pares donde participa alguna fantasma ---
pares_fantasma = df_pares[df_pares['ruc_a'].isin(rucs_fantasma) | df_pares['ruc_b'].isin(rucs_fantasma)]
print(f"\nPares que involucran a alguna fantasma: {len(pares_fantasma):,}")
print(pares_fantasma.sort_values('jaccard', ascending=False).head(20))

print("\nDistribución de Jaccard - pares con fantasma vs. resto:")
print("Con fantasma:", pares_fantasma['jaccard'].describe())
print("\nResto (muestra):", df_pares[~(df_pares['ruc_a'].isin(rucs_fantasma) | df_pares['ruc_b'].isin(rucs_fantasma))]['jaccard'].sample(min(50000, len(df_pares)), random_state=42).describe())

In [ ]:
# Inspección puntual del par más consistente estadísticamente
ruc_par = 'RUC_OCULTO_16'

info_ruc_par = con2.execute(f"""
    SELECT DISTINCT
        proveedor_ruc, proveedor_razon_social_sri, proveedor_estado_actual_sri,
        proveedor_antiguedad_dias, proveedor_es_fantasma
    FROM base_modelado_v13
    WHERE proveedor_ruc = '{ruc_par}'
""").fetchdf()
print(info_ruc_par)

# Procesos donde coincidieron ambos
procesos_en_comun = oferentes_validos[oferentes_validos['ocid'].isin(
    oferentes_validos[oferentes_validos['oferente_ruc'] == 'RUC_OCULTO_02']['ocid']
) & (oferentes_validos['oferente_ruc'] == ruc_par)]
print(f"\nProcesos en común: {len(procesos_en_comun)}")
print(procesos_en_comun[['ocid', 'anio']].head(10))

In [ ]:
import pandas as pd
import numpy as np

try:
    import xgboost as xgb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    import xgboost as xgb

from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

# =========================================================
# MODELO 3: XGBoost supervisado, target = proveedor_es_fantasma (SRI)
# =========================================================

df_m3 = con2.execute("""
    SELECT
        ocid, fecha_release,
        EXTRACT(YEAR FROM fecha_release) AS anio_proceso,
        tiene_periodo_competitivo,
        desviacion_precio_log_z,
        concentracion_entidad_proveedor,
        proveedor_antiguedad_dias,
        proveedor_es_entidad_publica,
        proveedor_contrato_tras_suspension,
        proveedor_provincia_distinta_entidad,
        proveedor_unico,
        n_oferentes,
        proveedor_es_fantasma
    FROM base_modelado_v13
    WHERE desviacion_precio_log_z IS NOT NULL
      AND concentracion_entidad_proveedor IS NOT NULL
""").fetchdf()

print(f"Filas totales: {len(df_m3):,}")
print(f"Distribución por año:\\n{df_m3['anio_proceso'].value_counts().sort_index()}")

# --- Manejo explícito de nulos (mismo criterio que Isolation Forest) ---
df_m3['antiguedad_desconocida'] = df_m3['proveedor_antiguedad_dias'].isna().astype(int)
df_m3['proveedor_antiguedad_dias_num'] = df_m3['proveedor_antiguedad_dias'].fillna(-1)
df_m3['proveedor_unico_desconocido'] = df_m3['proveedor_unico'].isna().astype(int)
df_m3['proveedor_unico_num'] = df_m3['proveedor_unico'].fillna(False).astype(int)
df_m3['n_oferentes'] = df_m3['n_oferentes'].fillna(0)

for col in ['tiene_periodo_competitivo', 'proveedor_es_entidad_publica',
            'proveedor_contrato_tras_suspension', 'proveedor_provincia_distinta_entidad']:
    df_m3[col] = df_m3[col].astype(int)

features_m3 = [
    'desviacion_precio_log_z', 'concentracion_entidad_proveedor',
    'proveedor_antiguedad_dias_num', 'antiguedad_desconocida',
    'proveedor_es_entidad_publica', 'proveedor_contrato_tras_suspension',
    'proveedor_provincia_distinta_entidad', 'tiene_periodo_competitivo',
    'proveedor_unico_num', 'proveedor_unico_desconocido', 'n_oferentes',
]

y = df_m3['proveedor_es_fantasma'].astype(int)
print(f"\\nCasos positivos totales: {y.sum()} de {len(y):,} ({y.mean()*100:.4f}%)")

# --- SPLIT TEMPORAL (no aleatorio) ---
ANIO_CORTE = 2023  # train: <2023 | test: >=2023 -- ajustable
train_mask = df_m3['anio_proceso'] < ANIO_CORTE
test_mask = df_m3['anio_proceso'] >= ANIO_CORTE

X_train, y_train = df_m3.loc[train_mask, features_m3], y[train_mask]
X_test, y_test = df_m3.loc[test_mask, features_m3], y[test_mask]

print(f"\\nTrain: {len(X_train):,} filas, {y_train.sum()} fantasmas ({y_train.mean()*100:.4f}%)")
print(f"Test:  {len(X_test):,} filas, {y_test.sum()} fantasmas ({y_test.mean()*100:.4f}%)")

In [ ]:
# --- Entrenamiento XGBoost con manejo de desbalance extremo ---
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

modelo_xgb = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
)
modelo_xgb.fit(X_train, y_train)

# --- Evaluación en TEST (años >= 2023, nunca vistos en entrenamiento) ---
y_score_test = modelo_xgb.predict_proba(X_test)[:, 1]

auc_roc = roc_auc_score(y_test, y_score_test)
auc_pr = average_precision_score(y_test, y_score_test)
print(f"\nAUC-ROC (test temporal): {auc_roc:.4f}")
print(f"PR-AUC (test temporal):  {auc_pr:.4f}   (línea base = tasa de positivos = {y_test.mean():.6f})")

# --- Precision@K: de los K procesos más riesgosos según el modelo, ¿cuántos son fantasma real? ---
df_test_resultado = X_test.copy()
df_test_resultado['y_real'] = y_test.values
df_test_resultado['score'] = y_score_test
df_test_resultado = df_test_resultado.sort_values('score', ascending=False)

for k in [10, 50, 100, 200, 500, 1000]:
    top_k = df_test_resultado.head(k)
    capturados = top_k['y_real'].sum()
    print(f"Top {k}: {capturados} de {int(y_test.sum())} fantasmas reales capturadas (precision={capturados/k:.4f})")

# --- Baseline de comparación: modelo con una sola variable (proveedor_unico) ---
auc_pr_baseline = average_precision_score(y_test, X_test['proveedor_unico_num'])
print(f"\nPR-AUC baseline (solo proveedor_unico): {auc_pr_baseline:.4f}")

# --- Importancia de variables ---
importancias = pd.Series(modelo_xgb.feature_importances_, index=features_m3).sort_values(ascending=False)
print("\nImportancia de variables:")
print(importancias)

In [ ]:
# =========================================================
# CORRECCIÓN: desviacion_precio_log_z sin fuga temporal
# (mediana/MAD calculados solo con años ANTERIORES al del proceso)
# =========================================================

con2.execute("""
    CREATE OR REPLACE TABLE base_precio AS
    SELECT ocid, anio_proceso, tipo_contratacion_canonico,
           CASE WHEN monto_final > 0 THEN LN(monto_final) ELSE NULL END AS log_monto
    FROM (
        SELECT *, EXTRACT(YEAR FROM fecha_release) AS anio_proceso
        FROM base_modelado_v13
    )
    WHERE log_monto IS NOT NULL
""")

anios = list(range(2016, 2027))  # 2015 no tiene años previos -> sin baseline, se documenta como limitación

con2.execute("CREATE OR REPLACE TABLE estadisticos_precio_expandido (anio_aplicable INT, tipo_contratacion_canonico VARCHAR, mediana_log_exp DOUBLE, mad_log_exp DOUBLE)")

for anio in anios:
    con2.execute(f"""
        INSERT INTO estadisticos_precio_expandido
        WITH previos AS (
            SELECT tipo_contratacion_canonico, log_monto
            FROM base_precio
            WHERE anio_proceso < {anio}
        ),
        medianas AS (
            SELECT tipo_contratacion_canonico, MEDIAN(log_monto) AS mediana_log_exp
            FROM previos GROUP BY 1
        )
        SELECT {anio} AS anio_aplicable, p.tipo_contratacion_canonico,
               m.mediana_log_exp,
               MEDIAN(ABS(p.log_monto - m.mediana_log_exp)) AS mad_log_exp
        FROM previos p
        JOIN medianas m USING (tipo_contratacion_canonico)
        GROUP BY 1, 2, 3
    """)

n_stats = con2.execute("SELECT COUNT(*) FROM estadisticos_precio_expandido").fetchone()[0]
print(f"Estadísticos expandidos calculados: {n_stats:,} combinaciones año x categoría")
print(con2.execute("SELECT * FROM estadisticos_precio_expandido ORDER BY anio_aplicable, tipo_contratacion_canonico LIMIT 10").fetchdf())

In [ ]:
# --- Aplicar el z-score corregido y comparar contra el original (con fuga) ---
con2.execute("""
    CREATE OR REPLACE TABLE base_modelado_v14 AS
    SELECT
        b.*,
        e.mediana_log_exp,
        e.mad_log_exp,
        0.6745 * (bp.log_monto - e.mediana_log_exp) / NULLIF(e.mad_log_exp, 0) AS desviacion_precio_log_z_sinfuga
    FROM (SELECT *, EXTRACT(YEAR FROM fecha_release) AS anio_proceso FROM base_modelado_v13) b
    LEFT JOIN base_precio bp ON b.ocid = bp.ocid
    LEFT JOIN estadisticos_precio_expandido e
        ON b.anio_proceso = e.anio_aplicable AND b.tipo_contratacion_canonico = e.tipo_contratacion_canonico
""")

n_v14 = con2.execute("SELECT COUNT(*) FROM base_modelado_v14").fetchone()[0]
print(f"base_modelado_v14: {n_v14:,} filas")

diag_comparacion = con2.execute("""
    SELECT
        COUNT(*) FILTER (WHERE desviacion_precio_log_z_sinfuga IS NULL) AS n_sin_baseline_2015,
        COUNT(*) FILTER (WHERE ABS(desviacion_precio_log_z_sinfuga) > 3.5) AS n_atipicos_sinfuga,
        COUNT(*) FILTER (WHERE ABS(desviacion_precio_log_z) > 3.5) AS n_atipicos_original_confuga
    FROM base_modelado_v14
""").fetchdf()
print(diag_comparacion)

# ¿Sigue el Modelo 1 capturando algo de las fantasmas con la versión sin fuga?
cruce_sinfuga = con2.execute("""
    SELECT proveedor_es_fantasma,
           COUNT(*) FILTER (WHERE ABS(desviacion_precio_log_z_sinfuga) > 3.5) AS n_atipico_sinfuga,
           COUNT(*) AS n_total
    FROM base_modelado_v14
    WHERE proveedor_es_fantasma IS NOT NULL
    GROUP BY 1
""").fetchdf()
print(cruce_sinfuga)

con2.execute("""
    COPY base_modelado_v14 TO 'C:/SERCOP_FINAL/03_resultados/base_modelado_v14.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print("Guardado: base_modelado_v14.parquet")